# QuantLibAAD quickstart: option Greeks in one adjoint sweep

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auto-differentiation/QuantLibAAD-Py/blob/main/notebooks/01_quickstart.ipynb)

[QuantLibAAD](https://github.com/auto-differentiation/QuantLibAAD-Py) is QuantLib for Python built with
[XAD](https://github.com/auto-differentiation/xad), a fast automatic differentiation (AD) library. You write
ordinary QuantLib code; when its inputs are XAD numbers, XAD records the calculation and can give the derivatives
of the result with respect to every input.

This notebook prices a European call with QuantLib and gets all its first-order Greeks:

1. **Adjoint mode**: all sensitivities from one recording and one backward sweep.
2. **Check** against QuantLib's own analytic Greeks.
3. **Check** against bumping the inputs.

In [1]:
%pip install -q QuantLibAAD

## A QuantLib pricer

A one-year call struck at 8 on a stock at 7, priced with the Black-Scholes analytic engine. Spot, dividend
yield, risk-free rate and volatility each enter through a `SimpleQuote`, the inputs we want sensitivities to.
The same function works with plain floats and with XAD numbers.

In [2]:
import QuantLibAAD as ql
from xad.adj_1st import Tape

today = ql.Date(15, ql.May, 1998)
ql.Settings.instance().evaluationDate = today
day_counter = ql.Actual365Fixed()

inputs = {"spot": 7.0, "dividend": 0.05, "rate": 0.05, "vol": 0.10}


def make_option(spot, dividend, rate, vol):
    def flat_curve(r):
        return ql.YieldTermStructureHandle(ql.FlatForward(today, ql.QuoteHandle(ql.SimpleQuote(r)), day_counter))

    process = ql.BlackScholesMertonProcess(
        ql.QuoteHandle(ql.SimpleQuote(spot)),
        flat_curve(dividend),
        flat_curve(rate),
        ql.BlackVolTermStructureHandle(
            ql.BlackConstantVol(today, ql.TARGET(), ql.QuoteHandle(ql.SimpleQuote(vol)), day_counter)),
    )
    option = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, 8.0),
                              ql.EuropeanExercise(ql.Date(17, ql.May, 1999)))
    option.setPricingEngine(ql.AnalyticEuropeanEngine(process))
    return option


def price_option(spot, dividend, rate, vol):
    return make_option(spot, dividend, rate, vol).NPV()


print(f"price = {price_option(**inputs):.10f}")

price = 0.0303344207


## 1. All Greeks in one adjoint sweep

Register the inputs on a tape, run the unchanged pricer, then seed the price and sweep the tape backwards once.
Every input's derivative comes out of that one sweep, however many inputs there are.

In [3]:
def price_and_greeks(inputs):
    args = {name: ql.Real(value) for name, value in inputs.items()}
    with Tape() as tape:
        tape.registerInputs(list(args.values()))
        tape.newRecording()
        npv = price_option(**args)
        tape.registerOutput(npv)
        npv.derivative = 1.0
        tape.computeAdjoints()
        return npv.value, {name: arg.derivative for name, arg in args.items()}


price, greeks = price_and_greeks(inputs)

print(f"price = {price:.10f}")
for name, greek in greeks.items():
    print(f"d price / d {name:8} = {greek:+.10f}")

price = 0.0303344207
d price / d spot     = +0.0950998668
d price / d dividend = -0.6693467337
d price / d rate     = +0.6388460970
d price / d vol      = +1.1714772740


## 2. Check against QuantLib's analytic Greeks

The analytic engine also returns closed-form Greeks: delta, dividend rho, rho and vega.

In [4]:
option = make_option(**inputs)
analytic = {"spot": option.delta(), "dividend": option.dividendRho(), "rate": option.rho(), "vol": option.vega()}

print(f"{'d price / d':12} {'XAD adjoint':>14} {'analytic':>14} {'abs diff':>10}")
for name in inputs:
    print(f"{name:12} {greeks[name]:14.10f} {analytic[name]:14.10f} {abs(greeks[name] - analytic[name]):10.1e}")

d price / d     XAD adjoint       analytic   abs diff
spot           0.0950998668   0.0950998668    6.4e-16
dividend      -0.6693467337  -0.6693467337    5.3e-15
rate           0.6388460970   0.6388460970    5.3e-15
vol            1.1714772740   1.1714772740    4.7e-15


## 3. Check against bumping

Central differences reprice twice per input and are accurate only to the bump size squared.

In [5]:
h = 1e-6
bumped = {}
for name in inputs:
    up, down = dict(inputs), dict(inputs)
    up[name] += h
    down[name] -= h
    bumped[name] = (price_option(**up) - price_option(**down)) / (2 * h)

print(f"{'d price / d':12} {'XAD adjoint':>14} {'bumping':>14} {'abs diff':>10}")
for name in inputs:
    print(f"{name:12} {greeks[name]:14.10f} {bumped[name]:14.10f} {abs(greeks[name] - bumped[name]):10.1e}")

d price / d     XAD adjoint        bumping   abs diff
spot           0.0950998668   0.0950998671    2.7e-10
dividend      -0.6693467337  -0.6693467338    9.0e-11
rate           0.6388460970   0.6388460971    8.3e-11
vol            1.1714772740   1.1714772741    4.5e-11


The adjoint Greeks match QuantLib's closed forms to machine precision, and bumping to its truncation and
rounding error. Bumping needs two extra pricings per input; the adjoint sweep gives them all at once, which
matters when a price depends on whole curves and surfaces.

## Next steps

- [Curve risk](https://colab.research.google.com/github/auto-differentiation/QuantLibAAD-Py/blob/main/notebooks/02_curve_risk.ipynb): a swap book's sensitivities to every quote of a bootstrapped curve in one sweep.
- [Python examples](https://github.com/auto-differentiation/QuantLibAAD-Py/tree/main/Python/examples) and the [QuantLibAAD documentation](https://auto-differentiation.github.io/quantlib-aad/python/).
- [XAD for Python](https://github.com/auto-differentiation/xad-py): automatic differentiation of your own Python code.